# Interactive charts with Plotly

Python stage, lesson P22b. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** build a chart with `plotly.express` and read it back as a `Figure`; give each store its own color and put the bars in the order you choose; control what the hover label says; save a chart as an interactive HTML file and as a still picture.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The charts are interactive here: move the mouse over them, drag to zoom, click a name in the legend.

## Setup

In [ ]:
# Run this cell first. It installs the two libraries at the versions of the lesson.
# If a message asks you to restart the session, do it, then run every cell from the first.
%pip install -q plotly==7.1.0 kaleido==1.5.2

In [ ]:
import math
import numbers
import os

import pandas as pd
import plotly.express as px
import plotly.io as pio

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False


def trace_color(trace):
    """The color a bar or line trace is drawn in."""
    return trace.marker.color if trace.type == "bar" else trace.line.color


def check_figure(label, name, values=None, colors=None, traces=None, categories=None, panels=None, barmode=None, title=False, hover=None, digits=2, traps=None):
    """Look inside the Figure called `name`: the values of each trace (to `digits` decimals), its colors, the order of traces and categories, the panels, the barmode, the title and the hover text."""
    fig = globals().get(name)
    if fig is None or not hasattr(fig, "data") or not hasattr(fig, "layout"):
        RESULTS[label] = None
        print(f"{label}: give {name} the Figure of your chart, then run this cell again.")
        return False
    problems = []
    try:
        got_values = {t.name: {(x.item() if hasattr(x, "item") else x): round(float(y), digits) for x, y in zip(t.x, t.y)} for t in fig.data}
        got_colors = {t.name: str(trace_color(t)).lower() for t in fig.data}
        got_traces = [t.name for t in fig.data]
        got_categories = list(fig.layout.xaxis.categoryarray or [])
        got_panels = len({t.xaxis for t in fig.data})
        if values is not None and not same(got_values, {k: {x: float(y) for x, y in v.items()} for k, v in values.items()}):  # the figure holds floats
            problems.append(f"the traces should hold {values!r}, but they hold {got_values!r}")
        if colors is not None and not same(got_colors, {k: v.lower() for k, v in colors.items()}):
            problems.append(f"the colors should be {colors!r}, but they are {got_colors!r}")
        if traces is not None and got_traces != traces:
            problems.append(f"the traces should come in the order {traces!r}, but they come as {got_traces!r}")
        if categories is not None and got_categories != categories:
            problems.append(f"the categories should be in the order {categories!r}, but they are {got_categories!r}")
        if panels is not None and got_panels != panels:
            problems.append(f"the chart should have {panels} panels, but it has {got_panels}")
        if barmode is not None and fig.layout.barmode != barmode:
            problems.append(f"the bars should be drawn with barmode {barmode!r}, but barmode is {fig.layout.barmode!r}")
        if title and not (fig.layout.title.text or "").strip():
            problems.append("the chart has no title")
        if hover is not None:
            for t in fig.data:
                if not all(part in (t.hovertemplate or "") for part in hover):
                    problems.append(f"the hover text of {t.name} should contain {hover!r}")
                    break
    except Exception as error:
        problems.append(f"the Figure could not be read ({type(error).__name__})")
    if not problems:
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. " + "; ".join(problems) + ".")
    for what, wrong, message in (traps or []):
        try:
            if same(got_colors if what == "colors" else got_categories, wrong):
                print("Hint:", message)
        except Exception:
            pass
    return False


# The colors of the site's figures, used for the stores and, later, the categories.
BLUE, MAGENTA, GOLD, GREEN = "#0072b2", "#d6479a", "#ad9100", "#00a07a"

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# One row for each order line with its amount, store, month and category, as built in lessons P13, P15 and P17.
# All 3,287 lines count: the one canceled order and order 2075 (quantity 20 where 2 was meant) are in.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])
sales = lines.merge(orders[["order_id", "store_id", "ordered_at"]], on="order_id", validate="many_to_one")
sales = sales.merge(products[["product_id", "category"]], on="product_id", validate="many_to_one")
sales = sales.assign(month=pd.to_datetime(sales["ordered_at"]).dt.month)
# The table of the charts: the sales and the number of orders of each store and month, one row for each pair, as in lesson P22a.
store_names = dict(zip(stores["store_id"], stores["store_name"]))
store_month = sales.groupby(["store_id", "month"])["amount"].sum().reset_index()
orders_in = sales.groupby(["store_id", "month"])["order_id"].nunique().reset_index(name="orders")
store_month = store_month.merge(orders_in, on=["store_id", "month"], validate="one_to_one")
store_month = store_month.assign(store=store_month["store_id"].map(store_names))

## Unit 1: px.bar returns a Figure

**Predict first.** How many traces will `three.data` hold, and how many bars will each trace have?

In [ ]:
three = px.bar(store_month, x="month", y="amount", color="store", barmode="group")
print(type(three).__name__, len(three.data))
print([(t.name, len(t.y)) for t in three.data])
three.show()

**Predict first.** `sales` has 3,287 rows, many for each category, and no `barmode` is given. How many traces will `raw.data` hold, how many values will the first one have, and what will `raw.layout.barmode` say?

In [ ]:
raw = px.bar(sales, x="category", y="amount")
print(len(raw.data), len(raw.data[0].y), raw.layout.barmode)

## Unit 2: colors and order you choose

**Predict first.** Campus is the second store of the chart of three stores. Will it keep its color when Old Town is left out? Say True or False.

In [ ]:
def colors_of(table):
    chart = px.bar(table, x="month", y="amount", color="store", barmode="group")
    return {t.name: t.marker.color for t in chart.data}


before = colors_of(store_month)
after = colors_of(store_month[store_month["store"] != "Old Town"])
print(before)
print(after)
print(before["Campus"] == after["Campus"])

**Predict first.** Which store will come first in `fixed.data`, and which color will Campus have?

In [ ]:
COLORS = {"Old Town": BLUE, "Campus": MAGENTA, "Harbor": GOLD}  # one color for each store, in every chart
two = store_month[store_month["store"] != "Old Town"]
fixed = px.bar(two, x="month", y="amount", color="store", barmode="group",
               color_discrete_map=COLORS, category_orders={"store": ["Harbor", "Campus"]})
print([(t.name, t.marker.color) for t in fixed.data])
fixed.show()

## Unit 3: what the hover label says

**Predict first.** The default template lists the columns you plotted. What will `hover_data` change in it, and which line will be new?

In [ ]:
plain = px.bar(store_month, x="month", y="amount", color="store", barmode="group")
print(plain.data[0].hovertemplate)
rich = px.bar(store_month, x="month", y="amount", color="store", barmode="group",
              color_discrete_map=COLORS, hover_data={"orders": True, "amount": ":,.2f"})
print(rich.data[0].hovertemplate)
rich.update_traces(hovertemplate="Month %{x}: $%{y:,.2f} from %{customdata[0]} orders")
print(rich.data[0].hovertemplate)
rich.show()

## Unit 4: save the chart

**Predict first.** Which of the three files will be the largest, and about how many times larger than the file with `include_plotlyjs="cdn"` will it be?

In [ ]:
rich.write_html("cafe-sales.html")
rich.write_html("cafe-sales-cdn.html", include_plotlyjs="cdn")
try:
    rich.write_image("cafe-sales.svg", width=520, height=340)  # needs Kaleido and a Chrome browser
except Exception as error:  # usually a missing Chrome: pio.get_chrome() installs one
    print("The picture was not saved:", error)
for name in ["cafe-sales.html", "cafe-sales-cdn.html", "cafe-sales.svg"]:
    print(name, f"{os.path.getsize(name):,} bytes" if os.path.exists(name) else "missing")

## Unit 5: Check: the stacks are the totals

**Predict first.** Do the stacks of the rows in `raw` add up to the totals that pandas computes from the order lines?

In [ ]:
totals = sales.groupby("category")["amount"].sum()
stacks = pd.Series(raw.data[0].y, index=raw.data[0].x).groupby(level=0).sum()
print(totals.round(2).to_dict())
print(stacks.round(2).tolist() == totals.round(2).tolist())

## Finish it

### Exercise 1: Finish it

**Task:** How did each category sell month by month? `category_month` holds the sales of each category and month, one row for each pair. The call below draws one line for each category, all in one panel. Change it so that each category gets a panel of its own, two panels in a row, and give the chart a title that states what it shows.

**Expected:** the Figure `fig_1`: four traces in four panels, each line in its category color, and a title.

In [ ]:
category_month = sales.groupby(["category", "month"])["amount"].sum().reset_index()  # given: one row for each category and month
CATEGORY_COLORS = {"coffee": BLUE, "tea": GREEN, "food": GOLD, "bakery": MAGENTA}  # given: one color for each category
fig_1 = px.line(category_month, x="month", y="amount", color="category", color_discrete_map=CATEGORY_COLORS)  # given: four lines in one panel
# change the call above so that each category gets its own panel, two in a row, then give the chart a title
fig_1.show()

In [ ]:
_ = check_figure("Exercise 1", "fig_1", values={
    "bakery": {1: 464.75, 2: 465.75, 3: 584.75, 4: 670.25, 5: 759.25, 6: 879.5},
    "coffee": {1: 1516.75, 2: 1311.75, 3: 1508.25, 4: 1940.25, 5: 2436.0, 6: 2094.0},
    "food": {1: 578.5, 2: 510.75, 3: 525.75, 4: 1034.25, 5: 1253.5, 6: 987.75},
    "tea": {1: 549.75, 2: 382.0, 3: 381.5, 4: 746.0, 5: 654.5, 6: 518.75},
}, colors=CATEGORY_COLORS, panels=4, title=True)

<details>
<summary>Hint</summary>

The argument that gives each value of a column its own panel is `facet_col`, and `facet_col_wrap` sets how many panels fit in a row. The title is the `title` argument.

</details>

<details>
<summary>Solution</summary>

```python
category_month = sales.groupby(["category", "month"])["amount"].sum().reset_index()
CATEGORY_COLORS = {"coffee": BLUE, "tea": GREEN, "food": GOLD, "bakery": MAGENTA}
fig_1 = px.line(category_month, x="month", y="amount", color="category", color_discrete_map=CATEGORY_COLORS,
                facet_col="category", facet_col_wrap=2, title="Coffee leads in every month and peaks in May")
fig_1.show()
```

The four lines are the monthly sales of each category: coffee 1,516.75, 1,311.75, 1,508.25, 1,940.25, 2,436.00 and 2,094.00. By default the axes of the panels are linked, the facet page says, so zooming in one panel zooms the others; all four lines stay in their colors.

</details>

## On your own

### Exercise 2: On your own

**Task:** Which cafe takes the largest share of its orders at the weekend? Before you start, say in one sentence what the chart shows: for each store, the percent of its orders placed on a Saturday or a Sunday, all 1,800 orders. Draw one bar for each store on a Figure called `fig_2`: each store in its color from `COLORS`, the stores in the order Old Town, Campus, Harbor, and a title that states the answer. Write the code in the empty cell of the notebook.

**Expected:** the Figure `fig_2`: three traces, one bar each, in the order Old Town, Campus, Harbor, in the colors blue, magenta and gold, each bar as high as the store's percent (compared to one decimal), and a title.

In [ ]:
_ = check_figure("Exercise 2", "fig_2", values={
    "Old Town": {"Old Town": 36.6}, "Campus": {"Campus": 34.6}, "Harbor": {"Harbor": 39.4},
}, digits=1, colors=COLORS, traces=["Old Town", "Campus", "Harbor"], categories=["Old Town", "Campus", "Harbor"], title=True,
    traps=[("colors", {"Old Town": "#00cc96", "Campus": "#636efa", "Harbor": "#ef553b"}, "Plotly gave the stores its default colors: pass color_discrete_map=COLORS."),
           ("categories", [], "The stores come in the order of the rows: pass category_orders for the store column.")])

<details>
<summary>Hint</summary>

`orders` has the time of each order and its `store_id`. `.dt.day_name()` (lesson P17) and `.isin([...])` give True or False for each order, and the mean of True and False values is the share of True: `groupby(...).mean()`, times 100. The colors and the order of the stores are arguments of `px.bar`.

</details>

<details>
<summary>Solution</summary>

```python
by_store = orders.assign(store=orders["store_id"].map(store_names),
                         weekend=pd.to_datetime(orders["ordered_at"]).dt.day_name().isin(["Saturday", "Sunday"]))
weekend_share = by_store.groupby("store")["weekend"].mean().mul(100).reset_index(name="weekend_percent")
fig_2 = px.bar(weekend_share, x="store", y="weekend_percent", color="store", color_discrete_map=COLORS,
               category_orders={"store": ["Old Town", "Campus", "Harbor"]},
               title="Harbor has the largest weekend share, 39.4%")
fig_2.show()
```

Harbor has 140 of its 355 orders at the weekend, 39.4%; Old Town 267 of 729, 36.6%; Campus 248 of 716, 34.6%. Old Town has the most weekend orders, but Harbor the largest share, with about half as many orders in all. Without `category_orders` the stores come in alphabetical order, the order `groupby` sorts them in, and without `color_discrete_map` in Plotly's own colors; the check cell flags both.

</details>

## Check yourself

1. Without looking back: a Plotly Express bar chart of order lines that share a category: are the bars means, like seaborn's, and what does the chart draw instead?

<details>
<summary>answer</summary>

No. Plotly Express draws one rectangle for each row, and rectangles that share a category stack on top of each other, so the stack is a total. Summarize first with `groupby(...).sum()` when you want one bar for each category with its own hover label.

</details>

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants Old Town in the same blue in every chart and wrote a color for each store, but in her chart Old Town is not blue. Fix the line so that Old Town, Campus and Harbor use the colors of `COLORS`.

**Expected:** the Figure `fig_3`: three traces in the order Old Town, Campus, Harbor, in the colors blue, magenta and gold, and a title.

In [ ]:
colleague_colors = {"Old town": BLUE, "Campus": MAGENTA, "Harbor": GOLD}  # the colleague's map
fig_3 = px.bar(store_month, x="month", y="orders", color="store", barmode="group", color_discrete_map=colleague_colors,
               title="Orders each month: Harbor starts in April")
fig_3.show()

In [ ]:
_ = check_figure("Exercise 3", "fig_3", colors=COLORS, traces=["Old Town", "Campus", "Harbor"], title=True,
                 traps=[("colors", {"Old Town": "#ab63fa", "Campus": MAGENTA, "Harbor": GOLD}, "Only Old Town has the wrong color: open the hint below.")])

<details>
<summary>Hint</summary>

Print `fig_3.data[0].name` and compare it, letter by letter, with the keys of `colleague_colors`.

</details>

<details>
<summary>Solution</summary>

```python
colleague_colors = {"Old Town": BLUE, "Campus": MAGENTA, "Harbor": GOLD}
fig_3 = px.bar(store_month, x="month", y="orders", color="store", barmode="group", color_discrete_map=colleague_colors,
               title="Orders each month: Harbor starts in April")
fig_3.show()
```

The key was `"Old town"`, with a small t, and the value in the table is `"Old Town"`. Plotly Express raised no error for a key that matches no value: in our run on plotly 7.1.0, Old Town took a purple, while Campus and Harbor kept theirs. The check cell compares each trace's color with `COLORS`.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")